# 03 — Phase 3: stage-2 GRPO (Simulation) from each stage-1 checkpoint

**Committed grid: 3 checkpoints x seed 42 = 3 cells** (plan §3 Phase 3). Do not start the stretch-goal seeds (43/44) until these 3 cells are complete, validated, and Phase 4 has produced a readable result from them — same 'commit the readable result before expanding' discipline as the 4070 plan's two-pass structure.

In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
REPO_URL = 'https://github.com/WYR186/RLVR.git'  # HTTPS; if the repo is
# private, authenticate interactively (git credential prompt / a token you
# paste when asked) rather than embedding a token in this notebook.
REPO_DIR = '/content/RLVR'
import os
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

EXP2_DIR = f'{REPO_DIR}/experiment 2'
sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path — pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

import json
from pathlib import Path
CONFIG = json.load(open(f'{EXP2_DIR}/exp2_colab_config.json'))
DATA_DIR = Path(EXP2_DIR) / 'data'
print('config loaded:', CONFIG['experiment'])

In [ ]:
import gc, torch
splits = json.loads((DATA_DIR / 'exp2_splits.json').read_text())
domain_field = splits['domain_field']
PROMPT_FIELD, ANSWER_FIELD = splits['prompt_field'], splits['answer_field']
raw = guru_data._load_raw(revision=splits['dataset_revision'])
train_split = raw['train'] if 'train' in raw else next(iter(raw.values()))
stage_b_pool = guru_data.filter_stage_subset(train_split, guru_data.STAGE_B_SUBSET_NAMES, domain_field)

stage_b_train_full = stage_b_pool.select(splits['stage_b_train_idx']).map(lambda ex: {
    'prompt': str(ex[PROMPT_FIELD]), 'answer': str(ex[ANSWER_FIELD])
}, remove_columns=stage_b_pool.column_names)
stage_b_eval = stage_b_pool.select(splits['stage_b_eval_idx'])
eval_prompts = [str(r[PROMPT_FIELD]) for r in stage_b_eval]
eval_golds = [str(r[ANSWER_FIELD]) for r in stage_b_eval]

RUN_DIR = f'{EXP2_DIR}/../eaaj-pilot/outputs/exp2_colab_guru_math7b_REPLACE_WITH_HASH'
STAGE_A_DIR = f'{RUN_DIR}/stage_a'
sb = CONFIG['stage_b']
sa = CONFIG['stage_a']

## Committed grid: 3 checkpoints x seed 42

In [ ]:
# eval interval = spacing of the pre-registered eval points [0,10,...,50]
EVAL_EVERY = sb['eval_at_updates'][1] - sb['eval_at_updates'][0]
assert EVAL_EVERY > 0

results = {}
for step in sa['adapt_from_checkpoints']:
    for seed in sb['committed_seeds']:
        cell_dir = f'{RUN_DIR}/stage_b/ckpt{step}_seed{seed}'
        print(f'=== ckpt {step}, seed {seed} ===')
        summary = pipeline.run_stage_b_adaptation(
            CONFIG['model_id'], CONFIG['peft'], f'{STAGE_A_DIR}/ckpt-{step}',
            stage_b_train_full, eval_prompts, eval_golds, cell_dir,
            budget_updates=sb['budget_updates'], eval_every=EVAL_EVERY,
            learning_rate=sb['learning_rate'], per_device_batch=sb['per_device_train_batch_size'],
            grad_accum=sb['gradient_accumulation_steps'], num_generations=sb['num_generations'],
            beta=sb['beta'], temperature=sb['temperature'], top_p=sb['top_p'],
            max_prompt_length=sb['max_prompt_length'], max_completion_length=sb['max_completion_length'],
            seed=seed)
        results[f'ckpt{step}_seed{seed}'] = summary
        print(summary)
        gc.collect(); torch.cuda.empty_cache()

print('Committed grid complete:', list(results.keys()))

## Stretch goal (only after the committed grid is complete and reported)

Uncomment and run only if Phase 0's measured throughput leaves runway before 2026-08-23 (plan §4 compute budget).

In [ ]:
# for step in sa['adapt_from_checkpoints']:
#     for seed in sb['stretch_goal_seeds']:
#         cell_dir = f'{RUN_DIR}/stage_b/ckpt{step}_seed{seed}'
#         summary = pipeline.run_stage_b_adaptation(
#             CONFIG['model_id'], CONFIG['peft'], f'{STAGE_A_DIR}/ckpt-{step}',
#             stage_b_train_full, eval_prompts, eval_golds, cell_dir,
#             budget_updates=sb['budget_updates'], eval_every=EVAL_EVERY,
#             learning_rate=sb['learning_rate'], per_device_batch=sb['per_device_train_batch_size'],
#             grad_accum=sb['gradient_accumulation_steps'], num_generations=sb['num_generations'],
#             beta=sb['beta'], temperature=sb['temperature'], top_p=sb['top_p'],
#             max_prompt_length=sb['max_prompt_length'], max_completion_length=sb['max_completion_length'],
#             seed=seed)
#         results[f'ckpt{step}_seed{seed}'] = summary
#         gc.collect(); torch.cuda.empty_cache()

## Commit reminder

Commit `stage_b/` (every cell's dashboard/curve/summary), prefix `exp2-colab:`, one commit per pass (committed grid, then stretch goal if run). Never overwrite a preserved `*_oom_*` directory.